## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## 做的时候一直低分，测试之后发现只能通过不能排序的示例，因此借助大模型和询问同学，发现是思路的问题，没有考虑好所有情况

import sys

LIMIT = 32768


class Solver:
    def __init__(self):
        self.n = 0
        self.A = 0
        self.B = 0
        self.blockSize = 0

        self.perm = []
        self.position = []
        self.operations = []

        self.failed = False

    def run(self):
        if not self.read_input():
            return

        self.init_block_size()

        if not self.prepare_base_permutation():
            print(-1)
            return

        if not self.restore_permutation():
            print(-1)
            return

        if self.failed or len(self.operations) > LIMIT:
            print(-1)
            return

        self.print_operations()

    def read_input(self):
        data = list(map(int, sys.stdin.buffer.read().split()))

        if len(data) < 3:
            print(-1)
            return False

        self.n = data[0]
        self.A = data[1]
        self.B = data[2]

        if len(data) != self.n + 3:
            print(-1)
            return False

        self.perm = data[3:]

        self.position = [0] * self.n
        self.rebuild_position()

        return True

    def rebuild_position(self):
        for i in range(self.n):
            self.position[self.perm[i]] = i

    def init_block_size(self):
        self.blockSize = (self.A - self.B + self.n) % self.n
        self.blockSize &= -self.blockSize

        if self.blockSize == 0:
            self.blockSize = self.n

    def add_operation(self, op):
        self.operations.append(op)

        if len(self.operations) > LIMIT:
            self.failed = True

    def apply_swap_magic(self):
        if self.failed:
            return

        self.add_operation(0)

        pa = self.position[self.A]
        pb = self.position[self.B]

        self.perm[pa], self.perm[pb] = self.perm[pb], self.perm[pa]
        self.position[self.A], self.position[self.B] = pb, pa

    def apply_add_magic(self, value):
        if self.failed:
            return

        value %= self.n

        if value < 0:
            value += self.n

        if value == 0:
            return

        self.add_operation(value)

        n = self.n
        perm = self.perm
        position = self.position

        for i in range(n):
            nv = perm[i] + value

            if nv >= n:
                nv -= n

            perm[i] = nv
            position[nv] = i

    def apply_xor_magic(self, value):
        if self.failed:
            return

        if value == 0:
            return

        self.add_operation(-value)

        n = self.n
        perm = self.perm
        position = self.position

        for i in range(n):
            nv = perm[i] ^ value
            perm[i] = nv
            position[nv] = i

    def get_pair_position(self, x, y):
        delta = (y - x + self.n - self.blockSize + self.n) % self.n

        px = 0
        py = 0

        step = self.n // 2

        while step >= 2 * self.blockSize:
            if delta >= step:
                delta -= step
                py += step // 2
            else:
                px += step // 2

            step >>= 1

        px += self.n // 2
        px += (x & (self.blockSize - 1))

        py += (x & (self.blockSize - 1))

        return px, py

    def swap_values(self, x, y):
        if self.failed:
            return

        groupX = (x // self.blockSize) % 2
        groupY = (y // self.blockSize) % 2

        if groupX == groupY:
            if groupX == 0:
                middle = (x & (self.blockSize - 1)) + self.blockSize
            else:
                middle = x & (self.blockSize - 1)

            self.swap_values(x, middle)
            self.swap_values(y, middle)
            self.swap_values(x, middle)
            return

        targetA, targetB = self.get_pair_position(self.A, self.B)
        targetX, targetY = self.get_pair_position(x, y)

        self.apply_add_magic((targetX - x + self.n) % self.n)
        self.apply_xor_magic(targetX ^ targetA)
        self.apply_add_magic((self.A - targetA + self.n) % self.n)

        self.apply_swap_magic()

        self.apply_add_magic((targetA - self.A + self.n) % self.n)
        self.apply_xor_magic(targetX ^ targetA)
        self.apply_add_magic((x - targetX + self.n) % self.n)

    def build_permutation(self, values, length):
        visited = [False] * length

        for i in range(length):
            if values[i] >= length:
                return False, []

            visited[values[i]] = True

        for i in range(length):
            if not visited[i]:
                return False, []

        if length == 1:
            return True, []

        half = length // 2

        left_values = [0] * half
        right_values = [0] * half

        for i in range(half):
            left_values[i] = values[i * 2] // 2
            right_values[i] = values[i * 2 + 1] // 2

        left_ok, left_ops = self.build_permutation(left_values, half)
        right_ok, right_ops = self.build_permutation(right_values, half)

        if not left_ok or not right_ok:
            return False, []

        ops = []

        if values[0] & 1:
            if length == 2:
                ops.append(1)
            else:
                ops.append(-1)

        left_xor = 0

        for op in left_ops:
            if op > 0:
                ops.append(-1)
                ops.append(1)
            else:
                ops.append(op * 2)
                left_xor ^= (-op) * 2

        if left_xor != 0:
            ops.append(-left_xor)

        right_xor = 0

        for op in right_ops:
            if op > 0:
                ops.append(1)
                ops.append(-1)
            else:
                ops.append(op * 2)
                right_xor ^= (-op) * 2

        if (right_xor & half) != (left_xor & half):
            return False, []

        if left_xor >= half:
            left_xor -= half

        if right_xor >= half:
            right_xor -= half

        if left_xor != right_xor:
            return False, []

        merged_ops = []

        for op in ops:
            if not merged_ops:
                merged_ops.append(op)
            elif op < 0 and merged_ops[-1] < 0:
                previous = -merged_ops[-1]
                current = -op
                merged = previous ^ current

                merged_ops[-1] = 0 if merged == 0 else -merged

                if merged_ops[-1] == 0:
                    merged_ops.pop()
            else:
                merged_ops.append(op)

        return True, merged_ops

    def prepare_base_permutation(self):
        if self.blockSize <= 1:
            return True

        base_values = [0] * self.blockSize

        for i in range(self.blockSize):
            base_values[i] = self.perm[i] & (self.blockSize - 1)

        ok, base_ops = self.build_permutation(base_values, self.blockSize)

        if not ok:
            return False

        for op in base_ops:
            if op > 0:
                self.apply_add_magic(op)
            else:
                self.apply_xor_magic(-op)

            if self.failed:
                return False

        return True

    def restore_permutation(self):
        for rem in range(self.blockSize):
            group_values = []

            for j in range(rem, self.n, self.blockSize):
                group_values.append(self.perm[j])

            group_values.sort()

            index = 0

            for j in range(rem, self.n, self.blockSize):
                if group_values[index] != j:
                    return False

                index += 1

            for j in range(rem, self.n, self.blockSize):
                if self.perm[j] != j:
                    self.swap_values(j, self.perm[j])

                    if self.failed:
                        return False

        return True

    def print_operations(self):
        print(len(self.operations))

        out = []

        for op in self.operations:
            if op == 0:
                out.append("0")
            elif op < 0:
                out.append(f"1 {-op}")
            else:
                out.append(f"2 {op}")

        sys.stdout.write("\n".join(out))

        if out:
            sys.stdout.write("\n")


def main():
    solver = Solver()
    solver.run()


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
from collections import deque


def can_finish(N, L, Maxn, S, stations):
    # 如果一开始体力就够，直接能到终点
    if Maxn >= L:
        return True

    # 同一个位置可能有多个补给站，只保留最便宜的
    cheapest = {}

    for p, c in stations:
        # 起点不用补给，终点也不用补给
        if 0 < p < L:
            if p not in cheapest:
                cheapest[p] = c
            else:
                cheapest[p] = min(cheapest[p], c)

    # 起点是 0，花费 0
    # 终点是 L，花费 0，因为到终点不需要补给
    points = [(0, 0)]

    for p, c in sorted(cheapest.items()):
        points.append((p, c))

    points.append((L, 0))

    m = len(points)

    INF = 10**18
    dp = [INF] * m
    dp[0] = 0

    # 队列里存的是点的下标
    # 它会保证队头永远是当前能用的、花费最少的点
    q = deque()
    q.append(0)

    for i in range(1, m):
        now_pos, now_cost = points[i]

        # 距离太远的点，已经跑不到当前位置了，扔掉
        while q and now_pos - points[q[0]][0] > Maxn:
            q.popleft()

        # 如果队列不空，说明可以从某个之前的点跑到这里
        if q:
            dp[i] = dp[q[0]] + now_cost

        # 花费超过 S 的方案，没有必要继续保留
        if dp[i] > S:
            dp[i] = INF

        # 如果这个点可达，就把它加入队列
        if dp[i] != INF:
            # 队尾如果比当前点更贵，那以后肯定不会优先用它
            # 所以可以删掉
            while q and dp[q[-1]] >= dp[i]:
                q.pop()

            q.append(i)

    # 最后一个点就是终点 L
    return dp[-1] <= S


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    idx = 0
    ans = []

    # 没有给 T，所以读到文件结束
    while idx < len(data):
        N = data[idx]
        L = data[idx + 1]
        Maxn = data[idx + 2]
        S = data[idx + 3]
        idx += 4

        stations = []
        for _ in range(N):
            p = data[idx]
            c = data[idx + 1]
            idx += 2
            stations.append((p, c))

        if can_finish(N, L, Maxn, S, stations):
            ans.append("Yes")
        else:
            ans.append("No")

    print("\n".join(ans))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## 一直超时，借助了一下大模型解决

import sys

MASK = (1 << 64) - 1
BASE = 1315423911
SEP = 35
LEFT = 1
RIGHT = 2

def make_t(s):
    n = len(s)
    t = bytearray(2 * n + 2)
    t[0] = LEFT
    t[1::2] = bytes([SEP]) * (n + 1)
    t[2::2] = s
    return bytes(t)

def manacher(t):
    m = len(t)
    arr = t + bytes([RIGHT])
    p = [0] * m
    mx = 0
    center = 0
    best = 0

    for i in range(1, m):
        if mx > i:
            mirror = center * 2 - i
            v = mx - i
            pv = p[mirror]
            pi = pv if pv < v else v
        else:
            pi = 1

        while arr[i + pi] == arr[i - pi]:
            pi += 1

        p[i] = pi

        x = i + pi
        if x > mx:
            mx = x
            center = i

        v = pi - 1
        if v > best:
            best = v

    return p, best

def solve(A, B):
    a = make_t(A)
    b = make_t(B)
    m = len(a)

    pa, besta = manacher(a)
    pb, bestb = manacher(b)

    power = [1] * (m + 1)
    pre = [0] * (m + 1)
    suf = [0] * (m + 1)

    base = BASE
    mask = MASK

    for i, x in enumerate(a):
        power[i + 1] = (power[i] * base) & mask
        pre[i + 1] = (pre[i] * base + x) & mask

    for i in range(m - 1, -1, -1):
        suf[i] = (suf[i + 1] * base + b[i]) & mask

    ans = besta if besta > bestb else bestb

    paL = pa
    pbL = pb
    aL = a
    bL = b
    preL = pre
    sufL = suf
    powL = power
    mm = m

    for i in range(2, mm):
        j = i - 2

        length = paL[i]
        v = pbL[j]
        if v > length:
            length = v

        sa = i - length + 1
        sb = j + length - 1

        hi = sa
        v = mm - 1 - sb
        if v < hi:
            hi = v

        if length + hi - 1 <= ans:
            continue

        if hi <= 0 or aL[sa - 1] != bL[sb + 1]:
            v = length - 1
            if v > ans:
                ans = v
            continue

        l2 = sb + 1
        add = 1
        lo = 2
        hi2 = hi

        while lo <= hi2:
            mid = (lo + hi2) >> 1
            l1 = sa - mid

            ha = (preL[sa] - ((preL[l1] * powL[mid]) & mask)) & mask
            hb = (sufL[l2] - ((sufL[l2 + mid] * powL[mid]) & mask)) & mask

            if ha == hb:
                add = mid
                lo = mid + 1
            else:
                hi2 = mid - 1

        v = length + add - 1
        if v > ans:
            ans = v

    return ans

def main():
    data = sys.stdin.buffer.read().split()
    n = int(data[0])
    A = data[1]
    B = data[2]
    print(solve(A, B))

if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
#coding:utf-8
# 每种优惠券同时最多拥有一张，但使用之后可以继续购买
import sys
from bisect import bisect_right
from array import array

MAXX=100000

def solve(m,ops,xs,qpos):
    #cnt[x]=0表示当前没有持有x
    #cnt[x]=1表示当前持有x
    cnt=bytearray(MAXX+1)

    #last[x]表示x上一次操作的位置
    last=array('i',[0])*(MAXX+1)

    qn=len(qpos)

    #并查集维护还没被使用的问号下标
    parent=array('i',range(qn+1))

    def find(x):
        #找当前可用问号
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    for i in range(1,m+1):
        op=ops[i]

        #问号本身先不处理
        if op==0:
            continue

        x=xs[i]

        if op==1:
            #当前已经持有x,再购买会冲突
            if cnt[x]==1:
                #找last[x]之后的第一个可用问号
                k=bisect_right(qpos,last[x])
                k=find(k)

                #没有可用问号,或者问号在当前行之后,无法修复
                if k==qn or qpos[k]>=i:
                    return i

                #这个问号补成O x
                parent[k]=find(k+1)

            else:
                cnt[x]=1

            #当前行成为x的最后一次操作
            last[x]=i

        else:
            #当前没有持有x,使用会冲突
            if cnt[x]==0:
                #找last[x]之后的第一个可用问号
                k=bisect_right(qpos,last[x])
                k=find(k)

                #没有可用问号,或者问号在当前行之后,无法修复
                if k==qn or qpos[k]>=i:
                    return i

                #这个问号补成I x
                parent[k]=find(k+1)

            else:
                cnt[x]=0

            #当前行成为x的最后一次操作
            last[x]=i

    return -1

def main():
    input=sys.stdin.buffer.readline
    ans=[]

    while True:
        line=input()

        if not line:
            break

        line=line.strip()

        if not line:
            continue

        m=int(line)

        ops=bytearray(m+1)
        xs=array('i',[0])*(m+1)
        qpos=array('i')

        for i in range(1,m+1):
            parts=input().split()

            #兼容英文?和中文？
            if parts[0]==b'I':
                ops[i]=1
                xs[i]=int(parts[1])
            elif parts[0]==b'O':
                ops[i]=2
                xs[i]=int(parts[1])
            else:
                ops[i]=0
                qpos.append(i)

        ans.append(str(solve(m,ops,xs,qpos)))

    sys.stdout.write("\n".join(ans))

if __name__=="__main__":
    main()

## E 任意点

In [ ]:
n = int(input())

points = []
for _ in range(n):
    x, y = map(int, input().split())
    points.append((x, y))

# 用来标记某个点有没有被访问过
visited = [False] * n


def dfs(i):
    visited[i] = True

    x1, y1 = points[i]

    # 看看第 i 个点能不能和其他点连起来
    for j in range(n):
        if not visited[j]:
            x2, y2 = points[j]

            # x 相同，说明在同一列
            # y 相同，说明在同一行
            if x1 == x2 or y1 == y2:
                dfs(j)


# 统计一共有多少个连通块
count = 0

for i in range(n):
    if not visited[i]:
        count += 1
        dfs(i)

# k 个连通块，最少加 k - 1 个点
print(count - 1)

## F 通配符匹配

In [ ]:
#coding:utf-8
# 短段带?直接用re，长段才用固定块锚点
import sys
import re

STAR=42
Q=63
DOT=46
REG_LIMIT=256

def build_seg(seg):
    #预处理一个不含星号的模式段
    l=len(seg)

    if Q not in seg:
        return (l,seg,None,None)

    if l<=REG_LIMIT:
        pat=bytes((DOT if x==Q else x) for x in seg)
        return (l,None,re.compile(pat),None)

    blocks=[]
    i=0

    while i<l:
        if seg[i]==Q:
            i+=1
        else:
            j=i
            while j<l and seg[j]!=Q:
                j+=1
            blocks.append((i,seg[i:j]))
            i=j

    return (l,None,None,blocks)

def match_at(seg,s,pos):
    #判断模式段是否从pos开始匹配
    l,exact,reg,blocks=seg

    if pos<0 or pos+l>len(s):
        return False

    if exact is not None:
        return s.startswith(exact,pos)

    if reg is not None:
        return reg.match(s,pos) is not None

    if not blocks:
        return True

    for off,blk in blocks:
        if not s.startswith(blk,pos+off):
            return False

    return True

def search_seg(seg,s,pos,right):
    #在[pos,right)内查找模式段
    l,exact,reg,blocks=seg

    if pos+l>right:
        return -1

    if l==0:
        return pos

    if exact is not None:
        k=s.find(exact,pos,right)
        if k==-1:
            return -1
        return k+l

    if reg is not None:
        m=reg.search(s,pos,right)
        if m is None:
            return -1
        return m.end()

    if not blocks:
        return pos+l

    best=0
    best_cnt=10**18

    #选择出现次数最少的固定块做锚点
    for idx,(off,blk) in enumerate(blocks):
        left=pos+off
        end=right-l+off+len(blk)

        if left>end:
            return -1

        first=s.find(blk,left,end)
        if first==-1:
            return -1

        cnt=s.count(blk,left,end)

        if cnt<best_cnt:
            best_cnt=cnt
            best=idx

    off,blk=blocks[best]
    left=pos+off
    end=right-l+off+len(blk)

    k=s.find(blk,left,end)

    while k!=-1:
        start=k-off
        ok=True

        for bo,bk in blocks:
            if bo==off and bk==blk:
                continue
            if not s.startswith(bk,start+bo):
                ok=False
                break

        if ok:
            return start+l

        k=s.find(blk,k+1,end)

    return -1

def make_checker(p):
    #计算最短需要长度
    min_len=len(p)-p.count(b'*')

    #没有星号时必须整串匹配
    if STAR not in p:
        seg=build_seg(p)
        need=len(p)

        def check_no_star(s):
            if len(s)!=need:
                return False
            return match_at(seg,s,0)

        return check_no_star

    parts=p.split(b'*')

    pre=build_seg(parts[0])
    suf=build_seg(parts[-1])

    mids=[]
    for x in parts[1:-1]:
        if x:
            mids.append(build_seg(x))

    pre_len=pre[0]
    suf_len=suf[0]

    def check(s):
        n=len(s)

        #长度不够直接失败
        if n<min_len:
            return False

        pos=0

        #第一段必须匹配开头
        if pre_len:
            if not match_at(pre,s,0):
                return False
            pos=pre_len

        #最后一段必须匹配结尾
        if suf_len:
            right=n-suf_len
            if right<pos:
                return False
            if not match_at(suf,s,right):
                return False
        else:
            right=n

        #中间段必须按顺序出现
        for seg in mids:
            pos=search_seg(seg,s,pos,right)
            if pos==-1:
                return False

        return pos<=right

    return check

def main():
    data=sys.stdin.buffer.read().split()

    p=data[0]
    n=int(data[1])
    names=data[2:2+n]

    check=make_checker(p)
    ans=[]

    for name in names:
        #判断当前文件名是否匹配
        if check(name):
            ans.append("YES")
        else:
            ans.append("NO")

    sys.stdout.write("\n".join(ans))

if __name__=="__main__":
    main()

## G 汉诺塔

In [ ]:
import sys


def solve():
    data = sys.stdin.read().split()

    n = int(data[0])
    priority = data[1:7]

    # 记录优先级
    rank = {}
    for i, op in enumerate(priority):
        rank[op] = i

    # 计算最小盘子在每根柱子上时，优先去哪里
    nxt = {}

    # 最小盘子在 A 上，只可能去 B 或 C
    if rank["AB"] < rank["AC"]:
        nxt["A"] = "B"
    else:
        nxt["A"] = "C"

    if rank["BA"] < rank["BC"]:
        nxt["B"] = "A"
    else:
        nxt["B"] = "C"

    if rank["CA"] < rank["CB"]:
        nxt["C"] = "A"
    else:
        nxt["C"] = "B"

    # 情况一：三个柱子形成一个完整的循环
    if (
        nxt["A"] == "B" and nxt["B"] == "C" and nxt["C"] == "A"
    ) or (
        nxt["A"] == "C" and nxt["C"] == "B" and nxt["B"] == "A"
    ):
        ans = 2 ** n - 1

    # 情况二：最小盘子在 B 和 C 之间来回走
    elif nxt["B"] == "C" and nxt["C"] == "B":
        ans = 3 ** (n - 1)

    # 情况三：最小盘子在 A 和另外一根柱子之间来回走
    else:
        ans = 2 * (3 ** (n - 1)) - 1

    print(ans)


if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
import sys


def knight_distance(xp, yp, xs, ys):
    # 先算两个点之间差了多少
    dx = abs(xs - xp)
    dy = abs(ys - yp)

    # 马步是对称的，让x表示较大的距离，y表示较小的距离
    x = max(dx, dy)
    y = min(dx, dy)

    # 原地不动
    if x == 0 and y == 0:
        return 0

    # 特殊情况：到旁边一格
    if x == 1 and y == 0:
        return 3

    # 特殊情况：到斜对角两格
    if x == 2 and y == 2:
        return 4

    # 至少需要这么多步：
    # 1. 最大方向每步最多走2步
    # 2. 两个方向总距离每步最多减少3步
    ans = max((x + 1) // 2, (x + y + 2) // 3)

    # 马每走一步，坐标和的奇偶性会变化一次
    # 所以步数的奇偶性必须和x+y对得上
    if (ans + x + y) % 2 == 1:
        ans += 1

    return ans


def main():
    data = list(map(int, sys.stdin.read().split()))

    xp, yp, xs, ys = data

    print(knight_distance(xp, yp, xs, ys))


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
class Solution:
    def largestRectangleArea(self,  heights: List[int]) -> int:
        #在左右两边各加一个0
        #左边的0方便算左边界
        #右边的0可以把栈里剩下的柱子全部结算掉
        heights = [0] + heights + [0]
 
        stack = []
        ans = 0
 
        for i in range(len(heights)):
            #如果当前柱子比栈顶柱子矮
            #说明栈顶柱子的右边界已经找到了
            while stack and heights[i] < heights[stack[-1]]:
                cur = stack.pop()
 
                #cur这根柱子的高度
                h = heights[cur]
 
                #弹出cur后，新的栈顶就是cur左边第一个比它矮的位置
                left = stack[-1]
 
                #当前位置i就是cur右边第一个比它矮的位置
                right = i
 
                #中间这些柱子高度都不低于h，所以可以形成矩形
                width = right - left - 1
 
                ans = max(ans, h * width)
 
            stack.append(i)
 
        return ans

## J 消防局的设立

In [ ]:
import sys


def solve():
    data = list(map(int, sys.stdin.buffer.read().split()))

    n = data[0]

    tree = [[] for _ in range(n)]

    #输入的第i个父亲，表示节点i+1和它的父亲相连
    #比如第1个数是节点2的父亲
    for i in range(2, n + 1):
        p = data[i - 1] - 1
        u = i - 1

        tree[u].append(p)
        tree[p].append(u)

    #先从1号点作为根，把父子关系整理出来
    parent = [-1] * n
    order = [0]
    parent[0] = -1

    for u in order:
        for v in tree[u]:
            if v != parent[u]:
                parent[v] = u
                order.append(v)

    #station[u]表示u这个点有没有消防局
    station = [False] * n

    #child_station[u]表示u的儿子里有多少个消防局
    child_station = [0] * n

    #grandchild_station[u]表示u的孙子里有多少个消防局
    grandchild_station = [0] * n

    def is_covered(u):
        #自己这里有消防局
        if station[u]:
            return True

        p = parent[u]

        if p != -1:
            #父节点有消防局
            if station[p]:
                return True

            #兄弟节点有消防局，距离也是2
            if child_station[p] > 0:
                return True

            gp = parent[p]

            #爷爷节点有消防局
            if gp != -1 and station[gp]:
                return True

        #儿子有消防局
        if child_station[u] > 0:
            return True

        #孙子有消防局
        if grandchild_station[u] > 0:
            return True

        return False

    ans = 0

    #反着处理，保证先处理下面的节点，再处理上面的节点
    for u in reversed(order):
        if is_covered(u):
            continue

        #如果u没有被覆盖，就往上走两步，尽量把消防局建高一点
        v = u

        if parent[v] != -1:
            v = parent[v]

        if parent[v] != -1:
            v = parent[v]

        #在v这里建消防局
        station[v] = True
        ans += 1

        #更新统计信息，方便后面快速判断某个点有没有被覆盖
        p = parent[v]

        if p != -1:
            child_station[p] += 1

            gp = parent[p]

            if gp != -1:
                grandchild_station[gp] += 1

    print(ans)


if __name__ == "__main__":
    solve()